# 인사 데이터 임베딩

### 라이브러리 설치

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 라이브러리 선언

In [2]:
# JSON 파일 읽기/쓰기 라이브러리
import json
# 파일 경로 처리 라이브러리
from pathlib import Path
# 운영체제 환경변수 읽기
import os
# .env 파일의 환경변수를 불러오는 라이브러리
from dotenv import load_dotenv
# 한국어 임베딩 모델 라이브러리
from sentence_transformers import SentenceTransformer, util

print('라이브러리 로딩 완료!')

C:\Users\heomingi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


라이브러리 로딩 완료!


### 환경 설정 *** 경로 확인 필요

In [3]:
# .env 파일의 설정값을 환경변수로 불러옴
load_dotenv()

# 청킹된 JSONL 파일이 있는 폴더 경로
INPUT_DIR  = Path(os.getenv('INPUT_DIR',  '../스플릿 및 청킹/output'))
# 임베딩 결과를 저장할 폴더 경로
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'output'))

# 출력 폴더가 없으면 자동으로 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 데이터구조정의서 기준 임베딩 모델 (384차원)
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL', 'paraphrase-multilingual-MiniLM-L12-v2')
# 한 번에 처리할 레코드 수 (배치 처리)
BATCH_SIZE      = int(os.getenv('BATCH_SIZE',   '64'))
# 데이터구조정의서 기준 벡터 차원 수
EXPECTED_DIM    = int(os.getenv('EXPECTED_DIM', '384'))

print(f'입력 디렉토리: {INPUT_DIR}')
print(f'출력 디렉토리: {OUTPUT_DIR}')
print(f'\n임베딩 모델: {EMBEDDING_MODEL}')
print(f'배치 크기  : {BATCH_SIZE}')
print(f'기대 차원  : {EXPECTED_DIM}')

입력 디렉토리: ..\스플릿 및 청킹\output
출력 디렉토리: output

임베딩 모델: paraphrase-multilingual-MiniLM-L12-v2
배치 크기  : 64
기대 차원  : 384


# 1. 데이터 로딩

### 1-1. JSONL 파일 로드

In [4]:
# 입력 폴더에서 .jsonl 파일 목록을 이름순으로 가져옴
jsonl_files = sorted(INPUT_DIR.glob('*.jsonl'))

if not jsonl_files:
    print(f'JSONL 파일 없음: {INPUT_DIR}')
    print('스플릿 및 청킹 노트북을 먼저 실행해 주세요.')
else:
    # 파일 이름을 키로, 레코드 리스트를 값으로 저장하는 딕셔너리
    datasets = {}
    for path in jsonl_files:
        records = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                records.append(json.loads(line.strip()))
        datasets[path.stem] = records
        print(f'  로딩: {path.name}  ({len(records):,}건)')

    print(f'\n로딩 완료! 총 {len(datasets)}개 파일')

  로딩: 급여정보_정제.jsonl  (2,000건)
  로딩: 기본인사정보_정제.jsonl  (4,090건)
  로딩: 역량성과_정제.jsonl  (2,000건)
  로딩: 통합인사정보_정제.jsonl  (7,035건)

로딩 완료! 총 4개 파일


# 2. 임베딩 모델 로딩

In [5]:
print(f'임베딩 모델 로딩 중: {EMBEDDING_MODEL}')
print('-' * 50)

# 한국어 지원 다국어 임베딩 모델 로딩
model = SentenceTransformer(EMBEDDING_MODEL)

print(f'모델 로딩 완료!')
print(f'  출력 차원: {model.get_sentence_embedding_dimension()}차원')

임베딩 모델 로딩 중: paraphrase-multilingual-MiniLM-L12-v2
--------------------------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7790.36it/s]

모델 로딩 완료!
  출력 차원: 384차원


C:\Users\heomingi\AppData\Local\Temp\claude\ipykernel_4384\1908194256.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'  출력 차원: {model.get_sentence_embedding_dimension()}차원')


# 3. 임베딩 생성

### 3-1. 배치 처리

In [6]:
print('임베딩 생성 시작...')
print('-' * 50)

for file_name, records in datasets.items():
    print(f'\n처리 중: [{file_name}]  총 {len(records):,}건')

    # embedding_text만 추출하여 리스트로 모음
    texts = []
    for record in records:
        texts.append(record['embedding_text'])

    # 배치 단위로 나눠서 임베딩 생성
    # batch_size: 한 번에 처리할 텍스트 수 (메모리 초과 방지)
    # show_progress_bar: 진행 상황 출력
    vectors = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # 각 레코드의 embedding_vector에 결과 저장
    error_count = 0
    for i in range(len(records)):
        try:
            # numpy 배열을 파이썬 리스트로 변환 (JSON 저장용)
            records[i]['embedding_vector'] = vectors[i].tolist()
        except Exception as e:
            # 에러 발생 시 빈 리스트 유지하고 카운트
            error_count += 1
            print(f'  에러: {records[i]["employee_id"]} — {e}')

    print(f'  완료: {len(records) - error_count:,}건 성공  /  에러: {error_count}건')

print('\n' + '-' * 50)
print('임베딩 생성 완료!')

임베딩 생성 시작...
--------------------------------------------------

처리 중: [급여정보_정제]  총 2,000건


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   3%|▎         | 1/32 [00:00<00:18,  1.64it/s]

Batches:   6%|▋         | 2/32 [00:01<00:18,  1.65it/s]

Batches:   9%|▉         | 3/32 [00:01<00:18,  1.60it/s]

...

Batches:  98%|█████████▊| 108/110 [01:28<00:00,  2.65it/s]

Batches:  99%|█████████▉| 109/110 [01:28<00:00,  2.63it/s]

Batches: 100%|██████████| 110/110 [01:28<00:00,  2.63it/s]

Batches: 100%|██████████| 110/110 [01:28<00:00,  1.24it/s]

  완료: 7,035건 성공  /  에러: 0건

--------------------------------------------------
임베딩 생성 완료!


# 4. 검증

### 4-1. 차원 수 일치 여부 확인

In [7]:
print('차원 수 일치 여부 확인 중...')
print('-' * 50)

# 데이터구조정의서 기준 차원 수
EXPECTED_DIM = 384

for file_name, records in datasets.items():
    mismatch_count = 0

    for record in records:
        dim = len(record['embedding_vector'])
        if dim != EXPECTED_DIM:
            mismatch_count += 1

    status = '정상' if mismatch_count == 0 else f'불일치 {mismatch_count}건'
    print(f'  {file_name}: [{status}]')

print('-' * 50)
print('차원 수 확인 완료!')

차원 수 일치 여부 확인 중...
--------------------------------------------------
  급여정보_정제: [정상]
  기본인사정보_정제: [정상]
  역량성과_정제: [정상]
  통합인사정보_정제: [정상]
--------------------------------------------------
차원 수 확인 완료!


### 4-2. 유사도 테스트

In [8]:
print('유사도 테스트 시작...')
print('의미적으로 가까운 레코드가 높은 유사도로 검색되는지 확인합니다.')
print('-' * 50)

# 기본인사정보로 테스트
if '기본인사정보_정제' in datasets:
    records = datasets['기본인사정보_정제']

    # 테스트 쿼리를 벡터로 변환
    test_query = '인사부 소속 대리'
    query_vector = model.encode(test_query)

    # 모든 레코드의 벡터를 리스트로 모음
    all_vectors = []
    for record in records:
        all_vectors.append(record['embedding_vector'])

    # 쿼리 벡터와 전체 레코드 벡터 간의 코사인 유사도 계산
    scores = util.cos_sim(query_vector, all_vectors)[0]

    # 유사도와 인덱스를 묶어서 리스트로 만들기
    score_list = []
    for i in range(len(scores)):
        score_list.append((float(scores[i]), i))

    # 유사도 높은 순으로 정렬
    score_list.sort(reverse=True)

    print(f'쿼리: "{test_query}"')
    print('\n상위 3개 결과:')
    print('-' * 50)
    for rank in range(3):
        score, idx = score_list[rank]
        emp_id = records[idx]['employee_id']
        text   = records[idx]['embedding_text']
        print(f'  {rank + 1}위 (유사도 {score:.3f}) {emp_id}: {text}')
    print('-' * 50)

print('유사도 테스트 완료!')

유사도 테스트 시작...
의미적으로 가까운 레코드가 높은 유사도로 검색되는지 확인합니다.
--------------------------------------------------


쿼리: "인사부 소속 대리"

상위 3개 결과:
--------------------------------------------------
  1위 (유사도 0.687) EMP1140: 대리 이전담당업무: 재무회계 회사명: 두리안정보기술 사업장위치: 서울 여의도 팀: 채용팀 직책: 팀장 이메일: kohaneul92@gmail
  2위 (유사도 0.676) EMP0981: 대리 이전담당업무: 인사관리 회사명: 두리안정보기술 사업장위치: 서울 여의도 팀: 전략기획팀 직책: 팀장 이메일: nohchaeyoung24@nate
  3위 (유사도 0.668) EMP0170: 이름: 유지후 부서: 기획부 직급: 차장 성별: 남 나이: 43 생년월일: 1984-12-07 주민등록번호: 841207-1218887 병역: 군필 입사일: 2015-01-07 근속기간: 11 학력: 전문대졸 출신대학: 영진전문대학교 학점: 2
--------------------------------------------------
유사도 테스트 완료!


# 5. 결과 저장

In [9]:
print('결과 저장 중...\n')

for file_name, records in datasets.items():
    out_path = OUTPUT_DIR / f'{file_name}.jsonl'

    with open(out_path, 'w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f'  저장: {out_path.name}  ({len(records):,}건)')

print('\n모든 파일 저장 완료!')

결과 저장 중...



  저장: 급여정보_정제.jsonl  (2,000건)


  저장: 기본인사정보_정제.jsonl  (4,090건)


  저장: 역량성과_정제.jsonl  (2,000건)


  저장: 통합인사정보_정제.jsonl  (7,035건)

모든 파일 저장 완료!
